In [1]:
from pathlib import Path
import os

# hunt for JSONL files anywhere reasonable
project_root = Path(r"C:\Users\shlok\projects\ddp-llm\parser")
for path in project_root.rglob("*.jsonl"):
    # skip venv and checkpoints
    if any(part in str(path) for part in [".venv", "checkpoints", ".ipynb_checkpoints"]):
        continue
    size_kb = path.stat().st_size / 1024
    print(f"{path.relative_to(project_root)}  ({size_kb:.1f} KB)")

data\seed.jsonl  (4.6 KB)
data\synthetic_raw.jsonl  (168.9 KB)
data\test.jsonl  (24.5 KB)
data\train.jsonl  (149.5 KB)


In [2]:
import json
import os
import re
import time
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm\parser")
DATA = PROJECT / "data"

# system prompt (same one your 3B fine-tune used)
SYSTEM_PROMPT = """You are a parser that converts a user's natural language response into a subset of the shown options.
The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.
Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")
Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.
Output ONLY the JSON. No explanation, no prose."""

# load seed and test
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

seed = load_jsonl(DATA / "seed.jsonl")
test = load_jsonl(DATA / "test.jsonl")

print(f"seed: {len(seed)} examples")
print(f"test: {len(test)} examples")
print(f"\nsample seed row: {seed[0]}")
print(f"\nsample test row: {test[0]}")

seed: 50 examples
test: 210 examples

sample seed row: {'options': ['A', 'B', 'C', 'D'], 'utterance': 'A looks right', 'label': ['A']}

sample test row: {'options': ['A', 'B', 'C', 'D'], 'utterance': 'except B, all are off', 'label': ['B'], 'category': 'negation'}


In [3]:
import random

# stratified 15-shot: at least one per category, rest random
def build_fewshot_examples(seed, n_shots=15, rng_seed=42):
    rng = random.Random(rng_seed)
    # bucket by "shape" of the answer to get category-like coverage from seed
    # seed doesn't have category tags, so we infer categories from label shape
    buckets = {"single": [], "multi": [], "reject_all": [], "star": []}
    for ex in seed:
        lbl = ex["label"]
        if lbl == "*":
            buckets["star"].append(ex)
        elif lbl == []:
            buckets["reject_all"].append(ex)
        elif isinstance(lbl, list) and len(lbl) == 1:
            buckets["single"].append(ex)
        else:
            buckets["multi"].append(ex)
    
    # ensure at least one from each non-empty bucket
    picked = []
    for k, v in buckets.items():
        if v:
            picked.append(rng.choice(v))
    # fill remaining with random samples from full seed (no repeats)
    remaining = [ex for ex in seed if ex not in picked]
    rng.shuffle(remaining)
    picked.extend(remaining[:max(0, n_shots - len(picked))])
    rng.shuffle(picked)
    return picked[:n_shots]

def format_example(ex, include_answer=True):
    """Format one example as a user/assistant turn pair."""
    user_msg = f"Options: {ex['options']}\nUser: {ex['utterance']}"
    parts = [{"role": "user", "content": user_msg}]
    if include_answer:
        parts.append({"role": "assistant", "content": json.dumps(ex["label"])})
    return parts

def build_prompt_messages(fewshot_examples, query_ex):
    """Full messages list for one query."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for demo in fewshot_examples:
        messages.extend(format_example(demo, include_answer=True))
    # final query, no assistant answer
    messages.extend(format_example(query_ex, include_answer=False))
    return messages

# sanity check
fewshot = build_fewshot_examples(seed, n_shots=15)
print(f"picked {len(fewshot)} few-shot examples")
print(f"first shot: options={fewshot[0]['options']}, utterance='{fewshot[0]['utterance']}', label={fewshot[0]['label']}")

# preview one full prompt
messages = build_prompt_messages(fewshot, test[0])
print(f"\ntotal messages in prompt: {len(messages)} (1 system + {len(fewshot)*2} demo + 1 query)")
print(f"\nlast 3 messages:")
for m in messages[-3:]:
    print(f"  [{m['role']}]: {m['content'][:80]}...")

picked 15 few-shot examples
first shot: options=['A', 'B', 'C', 'D'], utterance='none of these', label=[]

total messages in prompt: 32 (1 system + 30 demo + 1 query)

last 3 messages:
  [user]: Options: ['A', 'B', 'C', 'D']
User: option A for sure...
  [assistant]: ["A"]...
  [user]: Options: ['A', 'B', 'C', 'D']
User: except B, all are off...


In [4]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print(f"loading {MODEL_ID}...")
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

print(f"loaded in {time.time()-t0:.1f}s")
print(f"device: {next(model.parameters()).device}")
print(f"dtype: {next(model.parameters()).dtype}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

loading Qwen/Qwen2.5-3B-Instruct...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

loaded in 17.8s
device: cuda:0
dtype: torch.bfloat16
VRAM used: 6.17 GB


In [5]:
def parse_output(text):
    """Best-effort parse of model output into label shape."""
    text = text.strip()
    # try direct JSON parse first
    try:
        return json.loads(text)
    except:
        pass
    # look for JSON list pattern
    m = re.search(r'\[.*?\]', text)
    if m:
        try:
            return json.loads(m.group(0))
        except:
            pass
    # look for bare "*"
    if '"*"' in text or text.strip() == '*':
        return "*"
    # give up
    return None

def labels_match(pred, gold):
    """Set-equality for lists, string equality for '*'."""
    if pred is None:
        return False
    if gold == "*":
        return pred == "*"
    if isinstance(gold, list) and isinstance(pred, list):
        return set(pred) == set(gold)
    return False

@torch.no_grad()
def run_inference(query_ex, fewshot_examples):
    messages = build_prompt_messages(fewshot_examples, query_ex)
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated = out[0, inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

# fixed few-shot examples for whole eval (same across all test items — fair)
fewshot = build_fewshot_examples(seed, n_shots=15)

results = []
t0 = time.time()
for i, ex in enumerate(test):
    raw_output = run_inference(ex, fewshot)
    pred = parse_output(raw_output)
    correct = labels_match(pred, ex["label"])
    results.append({
        "idx": i,
        "utterance": ex["utterance"],
        "gold": ex["label"],
        "raw": raw_output,
        "pred": pred,
        "correct": correct,
        "category": ex.get("category", "unknown"),
        "format_valid": pred is not None,
    })
    if (i + 1) % 20 == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        eta = (len(test) - i - 1) / rate
        acc_so_far = sum(r["correct"] for r in results) / len(results)
        print(f"  {i+1}/{len(test)}   acc={acc_so_far:.1%}   {rate:.1f} ex/s   ETA {eta:.0f}s")

elapsed = time.time() - t0
print(f"\ndone in {elapsed:.1f}s ({len(test)/elapsed:.1f} ex/s)")

# overall stats
correct = sum(r["correct"] for r in results)
format_valid = sum(r["format_valid"] for r in results)
print(f"\n--- overall ---")
print(f"  exact match:    {correct}/{len(results)} = {correct/len(results):.1%}")
print(f"  format valid:   {format_valid}/{len(results)} = {format_valid/len(results):.1%}")

# per-category
from collections import defaultdict
by_cat = defaultdict(lambda: {"correct": 0, "total": 0})
for r in results:
    by_cat[r["category"]]["correct"] += int(r["correct"])
    by_cat[r["category"]]["total"] += 1

print(f"\n--- by category ---")
for cat in sorted(by_cat):
    stats = by_cat[cat]
    acc = stats["correct"] / stats["total"]
    print(f"  {cat:20s}  {stats['correct']:3d}/{stats['total']:3d}  ({acc:.1%})")

  20/210   acc=80.0%   1.6 ex/s   ETA 116s
  40/210   acc=70.0%   1.8 ex/s   ETA 92s
  60/210   acc=71.7%   1.9 ex/s   ETA 79s
  80/210   acc=77.5%   1.9 ex/s   ETA 67s
  100/210   acc=79.0%   2.0 ex/s   ETA 55s
  120/210   acc=80.8%   2.0 ex/s   ETA 44s
  140/210   acc=78.6%   2.1 ex/s   ETA 33s
  160/210   acc=79.4%   2.1 ex/s   ETA 24s
  180/210   acc=79.4%   2.1 ex/s   ETA 14s
  200/210   acc=79.0%   2.1 ex/s   ETA 5s

done in 99.7s (2.1 ex/s)

--- overall ---
  exact match:    166/210 = 79.0%
  format valid:   210/210 = 100.0%

--- by category ---
  comparative            24/ 26  (92.3%)
  mixed_sentiment        26/ 28  (92.9%)
  multi_positive         13/ 13  (100.0%)
  negation               43/ 61  (70.5%)
  off_topic              17/ 22  (77.3%)
  reject_all             29/ 29  (100.0%)
  single_positive         9/  9  (100.0%)
  uncertainty             5/ 22  (22.7%)
